In [ ]:
## Google Drive root path and project path definitions
DRIVE_ROOT_PATH = '/content/drive/MyDrive'
RLCCSAM_ROOT_PATH = f"{DRIVE_ROOT_PATH}/RL-CC-SAM"
PROJECT_ROOT = RLCCSAM_ROOT_PATH

In [ ]:
## Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")
## Ref: https://netraneupane.medium.com/how-to-install-libraries-permanently-in-google-colab-fb15a585d8a5

In [ ]:
###!source {DRIVE_ROOT_PATH}/colab_envs/llms/bin/activate && python -m pip install git+https://github.com/bowang-lab/MedSAM.git

In [ ]:
import os
import sys
from pathlib import Path

os.chdir(PROJECT_ROOT)
sys.path.append(PROJECT_ROOT)

SEGMENT_ANYTHING_DIR = Path(PROJECT_ROOT) / "segment-anything"
sys.path.insert(0, str(SEGMENT_ANYTHING_DIR))

In [ ]:
# %% environment and functions
import numpy as np
import matplotlib.pyplot as plt
import os
join = os.path.join
import torch
from segment_anything import sam_model_registry
from skimage import io, transform
import torch.nn.functional as F

# visualization functions
# source: https://github.com/facebookresearch/segment-anything/blob/main/notebooks/predictor_example.ipynb
# change color to avoid red and green
def show_mask(mask, ax, random_color=False):
    if random_color:
        color = np.concatenate([np.random.random(3), np.array([0.6])], axis=0)
    else:
        color = np.array([251/255, 252/255, 30/255, 0.6])
    h, w = mask.shape[-2:]
    mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    ax.imshow(mask_image)

def show_box(box, ax):
    x0, y0 = box[0], box[1]
    w, h = box[2] - box[0], box[3] - box[1]
    ax.add_patch(plt.Rectangle((x0, y0), w, h, edgecolor='blue', facecolor=(0,0,0,0), lw=2))

@torch.no_grad()
def medsam_inference(medsam_model, img_embed, box_1024, H, W):
    box_torch = torch.as_tensor(box_1024, dtype=torch.float, device=img_embed.device)
    if len(box_torch.shape) == 2:
        box_torch = box_torch[:, None, :] # (B, 1, 4)

    sparse_embeddings, dense_embeddings = medsam_model.prompt_encoder(
        points=None,
        boxes=box_torch,
        masks=None,
    )
    low_res_logits, _ = medsam_model.mask_decoder(
        image_embeddings=img_embed, # (B, 256, 64, 64)
        image_pe=medsam_model.prompt_encoder.get_dense_pe(), # (1, 256, 64, 64)
        sparse_prompt_embeddings=sparse_embeddings, # (B, 2, 256)
        dense_prompt_embeddings=dense_embeddings, # (B, 256, 64, 64)
        multimask_output=False,
        )

    low_res_pred = torch.sigmoid(low_res_logits)  # (1, 1, 256, 256)

    low_res_pred = F.interpolate(
        low_res_pred,
        size=(H, W),
        mode="bilinear",
        align_corners=False,
    )  # (1, 1, gt.shape)
    low_res_pred = low_res_pred.squeeze().cpu().numpy()  # (256, 256)
    medsam_seg = (low_res_pred > 0.5).astype(np.uint8)
    return medsam_seg


In [ ]:
# download model and data
###!!! !wget -O datasets/img_demo.png https://raw.githubusercontent.com/bowang-lab/MedSAM/main/assets/img_demo.png
## !wget -O medsam_vit_b.pth https://zenodo.org/records/10689643/files/medsam_vit_b.pth

In [ ]:
#%% load model and image
MedSAM_CKPT_PATH = f"{PROJECT_ROOT}/pretrained/medsam_vit_b.pth"
device = "cuda:0"
###!!! device = "cpu"
medsam_model = sam_model_registry['vit_b'](checkpoint=MedSAM_CKPT_PATH)
medsam_model = medsam_model.to(device)
medsam_model.eval()

img_np = io.imread('datasets/img_demo.png')
if len(img_np.shape) == 2:
    img_3c = np.repeat(img_np[:, :, None], 3, axis=-1)
else:
    img_3c = img_np
H, W, _ = img_3c.shape

In [ ]:
#%% image preprocessing and model inference
img_1024 = transform.resize(img_3c, (1024, 1024), order=3, preserve_range=True, anti_aliasing=True).astype(np.uint8)
img_1024 = (img_1024 - img_1024.min()) / np.clip(
    img_1024.max() - img_1024.min(), a_min=1e-8, a_max=None
)  # normalize to [0, 1], (H, W, 3)
# convert the shape to (3, H, W)
img_1024_tensor = torch.tensor(img_1024).float().permute(2, 0, 1).unsqueeze(0).to(device)

box_np = np.array([[95,255, 190, 350]])
# transfer box_np t0 1024x1024 scale
box_1024 = box_np / np.array([W, H, W, H]) * 1024
with torch.no_grad():
    image_embedding = medsam_model.image_encoder(img_1024_tensor) # (1, 256, 64, 64)

medsam_seg = medsam_inference(medsam_model, image_embedding, box_1024, H, W)

In [ ]:
print("img_1024:")
print(img_1024.shape)
print(img_1024.dtype)
print(img_1024.max())
print(img_1024.min())
print("img_1024_tensor:")
print(img_1024_tensor.shape)
print(img_1024_tensor.dtype)
print(img_1024_tensor.max())
print(img_1024_tensor.min())
print("box_np:")
print(box_np.shape)
print(box_np.dtype)
print(box_np.max())
print(box_np.min())
print("image_embedding:")
print(image_embedding.shape)
print(image_embedding.dtype)
print(image_embedding.max())
print(image_embedding.min())
print("box_1024:")
print(box_1024.shape)
print(box_1024.dtype)
print(box_1024.max())
print(box_1024.min())
print("medsam_seg:")
print(medsam_seg.shape)
print(medsam_seg.dtype)
print(medsam_seg.max())
print(medsam_seg.min())
print(H)
print(W)

In [ ]:
#%% visualize results
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(img_3c)
show_box(box_np[0], ax[0])
ax[0].set_title("Input Image and Bounding Box")
ax[1].imshow(img_3c)
show_mask(medsam_seg, ax[1])
show_box(box_np[0], ax[1])
ax[1].set_title("MedSAM Segmentation")
plt.show()